# Testing the `Intent_classification` n8n Workflow

Test cases are loaded from `unit_tests/classification/*/test.json`.
Each file contains a pre-built `input` dict and an `expectedOutput` with
`route_to` and a reference `narrative`.

### Routing values
`advisor` | `education` | `summary` | `unknown`

### Request payload shape
| Field | Meaning |
|---|---|
| `userMessage` | Latest message from the customer |
| `LastAImessage` | The previous agent's reply (if any) |
| `PrevAIagent` | Name of the agent that produced `LastAImessage` |
| `narrative` | Running narrative carried over from prior turns |

### URL note
- Test URL (n8n editor open): `{base}/webhook-test/{path}`
- Production URL (workflow Active): `{base}/webhook/{path}`

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

## 1. Configuration

In [ ]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "dbce5b9e-1397-459a-871a-5b27433f1640"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

## 2. Load test cases from JSON

In [ ]:
def _find_project_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "unit_tests").is_dir():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (directory containing 'unit_tests/')")


def load_unit_tests(agent_type: str) -> list[dict]:
    root = _find_project_root()
    test_dir = root / "unit_tests" / agent_type
    tests = []
    for tc_dir in sorted(test_dir.iterdir()):
        test_file = tc_dir / "test.json"
        if test_file.is_file():
            with open(test_file, encoding="utf-8") as f:
                tests.append(json.load(f))
    print(f"Loaded {len(tests)} test cases from {test_dir}")
    return tests


TEST_CASES = load_unit_tests("classification")
for tc in TEST_CASES:
    print(f"  {tc['testId']}: {tc['testDescription']}")

## 3. Webhook caller

In [ ]:
def call_intent_classifier(payload: dict, timeout: int = 30) -> dict:
    """POST the pre-built input dict to the Intent Classification webhook."""
    url = get_webhook_url()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Run a single example

Inspect one test case and confirm connectivity before running the full suite.

In [ ]:
sample_tc = TEST_CASES[0]
print(f"Test: {sample_tc['testId']} — {sample_tc['testDescription']}")
print("\nInput payload:")
print(json.dumps(sample_tc["input"], ensure_ascii=False, indent=2))
print("\nExpected output:")
print(json.dumps(sample_tc["expectedOutput"], ensure_ascii=False, indent=2))

# Uncomment once N8N_BASE_URL points to a real, reachable instance:
# result = call_intent_classifier(sample_tc["input"])
# print("\nActual output:")
# print(json.dumps(result, ensure_ascii=False, indent=2))

## 5. Run all test cases and compare `route_to`

In [ ]:
rows = []
for tc in TEST_CASES:
    expected = tc["expectedOutput"]
    expected_route = expected.get("route_to")
    try:
        result = call_intent_classifier(tc["input"])
        actual_route = result.get("route_to")
        actual_narrative = result.get("narrative", "")
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "expected_route": expected_route,
            "actual_route": actual_route,
            "route_match": actual_route == expected_route,
            "actual_narrative": actual_narrative,
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "expected_route": expected_route,
            "actual_route": None,
            "route_match": False,
            "actual_narrative": None,
            "error": str(exc),
        })

results_df = pd.DataFrame(rows)
summary_cols = ["testId", "scenario", "messageMode", "expected_route", "actual_route", "route_match", "error"]
print(f"Passed: {results_df['route_match'].sum()}/{len(results_df)}")
results_df[summary_cols]

## Notes

- This workflow is **Active**, so the production URL is
  `{base_url}/webhook/dbce5b9e-1397-459a-871a-5b27433f1640`.
- `route_to` is compared exactly; `narrative` is shown for inspection but not
  used for pass/fail since it is LLM-generated free text.
- Add more test cases by running the unit-test generator; they are picked up
  automatically the next time this notebook runs.
- If the n8n webhook requires authentication, add `auth=` or `headers=` to
  `requests.post` inside `call_intent_classifier`.